# Movie Recommender System - Approach

**Team Members:**
- Bekithemba Nkomo
- Peter Mangoro
- Masheia Dzimba

**Assignment:** Building a Hybrid Movie Recommender in Neo4j  
**Date:** March 8, 2026

---

## How We Will Tackle This Problem

Our strategy is to build the recommender system in **progressive layers**, starting simple and adding complexity:

### Phase 1: Foundation (Load & Enrich)
We will start by loading the three CSV files (users, movies, ratings) into Neo4j to create a basic bipartite graph: Users connected to Movies through RATED relationships. Then we will **enrich** this structure by extracting Genre and Director as first-class nodes. This is critical—instead of leaving genre/director as Movie properties, we will materialize them as nodes so we can leverage graph traversals for content-based filtering.

Why materialize Genre/Director nodes? Because it enables patterns like: "find movies that share a genre with movies this user rated highly." As properties, we would need slow property matching. As nodes, we can traverse the graph naturally.

### Phase 2: Exploration (EDA Queries)
Before computing anything, we need to understand what we are working with. The 8 EDA queries will reveal:
- **Sparsity:** With 20 users, 25 movies, and **101** `RATED` relationships in our CSV (the assignment brief mentions 100; we document the extra row in the main notebook), the user×movie grid is still **~80% sparse** (101 of 500 possible cells).
- **Popularity bias:** Are a few movies getting most of the ratings? (Long-tail distribution)
- **Rating inflation:** Do users mostly give 4-5 stars, or is the scale well-used?
- **Genre clustering:** Are there clear genre preferences that segment users?

These patterns directly inform our algorithm choices. For instance, if ratings are inflated (everyone gives 4-5 stars), magnitude-based similarity (cosine) might not add much value over set-based similarity (Jaccard).

### Phase 3: Collaborative Filtering (GDS Algorithms)
We will compute user-user similarity two ways:

**Jaccard (unweighted):** Treats ratings as binary—did they rate it or not? This finds users with overlapping taste regardless of whether they loved or hated the same movies. Simple, interpretable, but loses information.

**FastRP + kNN (weighted):** FastRP generates embeddings that capture the weighted rating neighborhood (a 5-star rating contributes more than a 1-star). Then kNN computes cosine similarity in embedding space. This is magnitude-aware—users who both loved {M1, M2} are more similar than users who both rated {M1, M2} but disagreed on the ratings.

We will write both similarity types back to the graph as relationships (SIMILAR_TASTE, KNN_SIMILAR) so we can reuse them for recommendations.

### Phase 4: Hybrid Recommendations (Collaborative + Content)
Pure collaborative filtering has weaknesses:
- **Cold-start:** New users/items with few ratings cannot be matched
- **Popularity bias:** Only recommends what is already popular
- **No diversity:** Stays in the user comfort zone

So we will implement a **hybrid scoring function** (as in our Jupyter notebook)—not a fixed 70/30 split, but a **single ranking score** that stays **collaborative-first** and adds **interpretable content boosts**:
- **Collaborative base:** For each candidate movie, aggregate **how many similar users** rated it highly (≥4) and their **average supporting rating**.
- **Content boosts:** Add weighted terms for **genre overlap** and **director overlap** between the candidate and movies the target user already rated ≥4 (tunable weights, e.g. `genreWeight`, `directorWeight`, with an optional **director-affinity** multiplier in Extension 4).

This keeps the strongest signal from neighbors while nudging rankings toward titles that align with the user’s demonstrated genre/director tastes—improving explainability ("recommended because you liked other films in this genre / by this director") without mixing separate 70% and 30% pipelines.

### Phase 5: Evaluation & Iteration
We will test the system using hold-out validation: hide 2 of a user ratings, generate recommendations, check if the hidden movies appear in the top K. We will also run all 4 extensions (cutoff sensitivity, cold-start handling, algorithm comparison, director affinity) to stress-test the approach.

---

## Selected Analytical Questions

We will answer **Question 1 (Taste Overlap Without Algorithms)** and **Question 2 (Genre Preference Profiles)**.

### Why Question 1: Taste Overlap Without Algorithms?

**What it does:** Uses pure Cypher to identify user pairs with the highest co-rated overlap and computes average absolute rating disagreement on those shared titles.

**Why we chose it:**
- **Establishes the collaborative baseline:** It shows that overlap count alone does not guarantee aligned preferences, which motivates moving from raw overlap to algorithmic similarity.
- **Connects directly to GDS Task 1 and Task 2:** It sets up the transition from binary co-engagement (Jaccard) to magnitude-aware similarity (FastRP + kNN/cosine).
- **Improves interpretation quality:** By examining both overlap and disagreement, we avoid over-claiming similarity from shared watch history.

**Expected insights:** We expect to find several high-overlap pairs with materially different rating behavior, demonstrating why weighted/embedding approaches are needed.

---

### Why Question 2: Genre Preference Profiles?

**What it does:** Computes genre preference profiles per user and compares profile similarity to a target user.

**Why we chose it:**
- **Validates the content signal used in hybrid scoring:** Our recommender adds genre/director overlap boosts; profile similarity confirms these features carry meaningful preference structure.
- **Complements collaborative methods in sparse settings:** Users can be close by genre tastes even when direct co-rating overlap is weak.
- **Supports explainability and cold-start fallback:** Genre-based affinity gives interpretable recommendations when collaborative evidence is limited.

**Expected insights:** We expect genre-profile similarity clusters that are related to, but not identical with, collaborative clusters, helping tune hybrid weights.

---

### How These Questions Support Our Overall Approach

Together, Q1 and Q2 provide the exact bridge used in the final notebook:
- **Q1 (overlap + disagreement)** explains limits of naive collaborative assumptions.
- **Q2 (genre profiles)** validates content-based structure for hybrid enrichment.

This pairing aligns with the submitted notebook narrative and the later GDS progression from Jaccard to FastRP+kNN to hybrid recommendations.

---

## Anticipated Challenges

### Challenge 1: Graph Sparsity
**The problem:** **101** ratings across 500 possible user-movie pairs ≈ **80% sparse** matrix. Most user pairs will not have any co-rated movies.

**Why this matters:**
- Jaccard similarity will be 0 for most user pairs (no overlap)
- Community detection might produce many singleton clusters
- Recommendations might be limited to only the most popular movies

**How we will handle it:**
- Use **lower similarity cutoffs** (0.2-0.3 instead of 0.5) to capture weak but meaningful connections
- Rely on **content signals** (genre, director) to fill gaps where collaborative signals fail
- **Hybrid scoring** ensures that even if collaborative filtering is weak, content matching can still recommend relevant items
- Be honest in evaluation: acknowledge that results are exploratory given the small sample

---

### Challenge 2: Jaccard vs Cosine—Which to Use?
**The dilemma:** Jaccard and cosine capture different things, and it is not obvious which is better.

**Example where they diverge:**
- User A rates {M1: 5, M2: 5, M3: 5}
- User B rates {M1: 1, M2: 1, M3: 1}
- Jaccard similarity = 1.0 (perfect overlap!)
- Cosine similarity = -1.0 (complete disagreement!)

Jaccard finds users who rated the same movies. Cosine finds users who rated them similarly.

**How we will handle it:**
- Implement **both** and compare results explicitly (Extension 3)
- Use Jaccard for initial exploration (simpler, easier to interpret)
- Use cosine/FastRP for final recommendations (captures agreement, not just overlap)
- Document where rankings diverge and explain why

---

### Challenge 3: Evaluation Is a Toy Test
**The problem:** Hold-out testing on one user with 2 hidden ratings is not statistically meaningful.

**Limitations we acknowledge upfront:**
- Cannot compute confidence intervals or significance tests
- Results could be luck (which 2 ratings were hidden)
- No baseline comparison (is our system better than "just recommend popular movies"?)

**How we will handle it:**
- Run hold-out on **multiple users** (3-5) to see if results generalize
- **Qualitative assessment:** Manually inspect recommendations—do they make sense given the user taste?
- **Explicitly state limitations** in the write-up (this is proof-of-concept, not production validation)
- **Discuss what production would look like:** k-fold cross-validation, A/B testing, precision@K, recall@K, NDCG, comparison to baselines

---

### Challenge 4: Cold-Start Problem
**The problem:** New users with 0-1 ratings have no collaborative filtering signal.

**Example (Extension 2):** We create user **U021** who has only rated **Inception** (5 stars). Pure collaborative and our default hybrid pipeline may return **no** candidates when neighbor support is empty.

**How we will handle it:**
- **Content-based fallback:** Recommend movies sharing **genre/director** with the seed highly-rated movie(s) (in our notebook: paths from Inception’s Sci‑Fi/Thriller and Christopher Nolan)
- **Popularity baseline:** Default to highest-rated movies overall
- **Demographic init:** Recommend popular movies within the user age group or occupation

Cold-start is where hybrid systems shine. Demonstrating graceful degradation (content-based works when collaborative fails) shows robustness.

---

### Challenge 5: Dual-Genre Weighting
**The problem:** Movies have up to 2 genres. How do we score genre matches?

**Ambiguous scenarios:**
- User loves Sci-Fi. Candidate A is {Sci-Fi, Action}. Candidate B is {Sci-Fi, Horror}. Same score?
- User rated 3 pure Sci-Fi movies highly. Should {Sci-Fi, Thriller} get full credit or partial?

**How we will handle it:**
- Treat both genre1 and genre2 equally: either match counts toward overlap
- In Cypher we count **distinct** overlapping genres/directors and multiply by tunable weights (e.g. `genreWeight` / `directorWeight` in the notebook—conceptually similar to +0.5 per genre match when `genreWeight=0.5`)
- **Acknowledge limitation:** Dual genres blur content signals; production systems would use TF-IDF on multi-hot genre vectors

---

### Challenge 6: GDS Projection Design
**The problem:** Different algorithms need different graphs.

**Examples:**
- **Node Similarity:** Needs bipartite User-Movie graph
- **Community Detection:** Needs monopartite User-User graph
- **FastRP:** Needs weighted relationships

**How we will handle it:**
- Create **task-specific projections** with descriptive names:
  - `user-movie-unweighted` (for Jaccard)
  - `user-movie-weighted` (for FastRP)
  - `user-user-similarity` (for Louvain)
- **Document each projection design:** which nodes, which relationships, weighted/unweighted, directed/undirected, and why
- Use `gds.graph.drop()` to clean up between tasks

---

### Challenge 7: Parameter Tuning Without Ground Truth
**The problem:** How do we know if `similarityCutoff=0.3` is better than `0.5`? Or `embeddingDimension=64` vs `128`?

**How we will handle it:**
- **Sensitivity analysis** (Extension 1): Test cutoffs {0.1, 0.3, 0.5} and report:
  - Number of similarity edges created (graph density)
  - Number of recommendations generated (coverage)
  - Qualitative assessment (do results look reasonable?)
- **Use literature defaults:** FastRP paper suggests embeddingDimension=64-128 for small graphs
- **Justify in write-up:** Explain every parameter choice, even if it is an educated guess

---

## Why This Approach Makes Sense

### Connection to MMDS Chapter 9
The textbook models recommendations as a sparse utility matrix (users × items). Neo4j represents this as a **bipartite graph**—relationships ARE the non-blank matrix entries. This is not just a different data structure; it is a better conceptual model:

- **Collaborative filtering** = graph traversal ("find users who rated similar movies")
- **Content-based filtering** = metadata integration (Genre/Director nodes)
- **Similarity computation** = GDS algorithms (Jaccard, FastRP, kNN)

The chapter distinguishes Jaccard (binary overlap) from cosine (magnitude-aware). Our approach implements both so we can compare them empirically.

### Hybrid > Pure Collaborative Filtering
Chapter 9 notes that real systems combine collaborative and content-based approaches. Our **hybrid ranking** (collaborative support + weighted genre/director overlap) balances:
- **Proven effectiveness** (collaborative filtering works)
- **Diversity & cold-start resilience** (content signals fill gaps)
- **Explainability** ("because you liked other Nolan films" > "users similar to you liked this")

### Progressive Complexity
We are starting simple (Jaccard, pure collaborative) and layering sophistication (FastRP, hybrid, community detection). This mirrors real-world development: build a baseline, measure it, improve iteratively.

---

## Next Steps

Once the approach is approved, we will:

1. **Set up Neo4j Desktop** and load the 3 CSVs
2. **Run graph enrichment** (create Genre/Director nodes)
3. **Complete all 8 EDA queries** with commentary
4. **Implement analytical questions** — at minimum **Q2 (Genre Preference Profiles)** and **Q3 (Long-Tail Analysis)**; our submitted notebook also includes **Q1 (Taste Overlap)** and **Q4 (Director–Genre Co-Occurrence)** for completeness
5. **Execute all 5 GDS tasks** (Jaccard, FastRP+kNN, recommendations, hybrid, community detection)
6. **Complete all 4 extensions**
7. **Polish the notebook** (visualizations, clean formatting, proofread)

Estimated time: 12-15 hours across 2 weeks.

---